# Section 7 — Encoding a Subsequent Tree: the Inductive Step

This notebook tests the implementation of **Section 7**:
1. **Definition 8 (Staged state & R_t)**: Incoming stage-(t-1) variables vs outgoing stage-t variables, and remaining trees R_t = D - 1 - t.
2. **Definition 9 (Clinching predicate Clinch_s^(t))**: Earliest certification bound R_t <= W - 1 and offset comparator LC^s >= LC^l + R_t.
3. **Definition 10 (Subsequent-tree formula Phi_t)**: The guarded choice between Hold (latching winner and freeing counters) and Step_t.

In [1]:
import os
import sys
import numpy as np
import dd.autoref as _bdd

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from rfbdd.forest import Tree, Forest
from rfbdd.discretize import discretize
from rfbdd.first_tree import compute_counter_width_m, declare_counter_and_win_vars
from rfbdd.subsequent_tree import (
    encode_geq_offset,
    encode_clinch,
    encode_counter_copy,
    encode_counter_increment,
    encode_hold,
    encode_subsequent_tree,
)

def check(condition: bool, message: str) -> None:
    status = 'PASS' if condition else 'FAIL'
    print(f'[{status}] {message}')
    assert condition, message

print('Setup complete.')

Setup complete.


## 7.1 Testing Offset Comparator LC^s >= LC^l + R_t (Definition 9)

Counters never exceed W - 1 in practice. For D = 2 (W = 2, M = 1, counter domain {0, 1}) and R_t = 1, we verify all valid counter pairs.

In [2]:
bdd = _bdd.BDD()
M = 1
declare_counter_and_win_vars(bdd, n_classes=2, m=M, with_barred=True)

comp_bdd = encode_geq_offset(bdd, label_s=0, label_l=1, r_t=1, m=M)

all_ok = True
for vs in range(2):
    for vl in range(2):
        assign = {
            'lc_0_0': bool(vs & 1), 'lc_0_1': bool((vs >> 1) & 1),
            'lc_1_0': bool(vl & 1), 'lc_1_1': bool((vl >> 1) & 1),
        }
        expected = (vs >= vl + 1)
        actual = (bdd.let(assign, comp_bdd) == bdd.true)
        if actual != expected:
            all_ok = False

check(all_ok, 'Offset comparator LC^s >= LC^l + 1 correct on all valid counter states')

[PASS] Offset comparator LC^s >= LC^l + 1 correct on all valid counter states


## 7.2 Building and Testing Phi_1 for Tree T_1 (Definition 10)

At tree t = 1, R_1 = 0. We verify that if T1 votes for class 0 with incoming state LC^0 = 1, LC^1 = 0, class 0 clinches and sets win_bar_0 = True.

In [3]:
t0 = Tree(
    feature=np.array([0, -1, 1, -1, -1], dtype=np.int64),
    threshold=np.array([5.7, np.nan, 10.0, np.nan, np.nan], dtype=np.float64),
    left=np.array([1, -1, 3, -1, -1], dtype=np.int64),
    right=np.array([2, -1, 4, -1, -1], dtype=np.int64),
    label=np.array([-1, 0, -1, 0, 1], dtype=np.int64),
)
t1 = Tree(
    feature=np.array([0, -1, 0, -1, -1], dtype=np.int64),
    threshold=np.array([2.5, np.nan, 8.1, np.nan, np.nan], dtype=np.float64),
    left=np.array([1, -1, 3, -1, -1], dtype=np.int64),
    right=np.array([2, -1, 4, -1, -1], dtype=np.int64),
    label=np.array([-1, 0, -1, 1, 0], dtype=np.int64),
)
forest = Forest(trees=[t0, t1], n_features=2, n_classes=2, classes=np.array([0, 1]), name='toy_forest')
disc = discretize(forest)

for var_name in disc.all_bit_names:
    bdd.declare(var_name)

phi_1 = encode_subsequent_tree(bdd, forest, disc, t_idx=1)
check(phi_1 != bdd.false, 'Phi_1 is satisfiable')

# Scenario 1: Input in cell 0 on feature 0 (x_0 = 1.0 <= 2.5)
sample_input = disc.encode_sample(np.array([1.0, 5.0]))
incoming_state = {
    'lc_0_0': True, 'lc_0_1': False,
    'lc_1_0': False, 'lc_1_1': False,
    'win_0': False, 'win_1': False,
}
outgoing_win0 = {
    'win_bar_0': True, 'win_bar_1': False,
}
res1 = bdd.let(sample_input | incoming_state | outgoing_win0, phi_1)
check(res1 == bdd.true, 'Scenario 1: Class 0 clinches the win at T1')

# Scenario 2: If someone had already won, Hold latches the win
incoming_won = {
    'win_0': True, 'win_1': False,
}
outgoing_hold = {
    'win_bar_0': True, 'win_bar_1': False,
}
res2 = bdd.let(incoming_won | outgoing_hold, phi_1)
check(res2 == bdd.true, 'Scenario 2: Hold branch successfully latches win_0')

[PASS] Phi_1 is satisfiable
[PASS] Scenario 1: Class 0 clinches the win at T1
[PASS] Scenario 2: Hold branch successfully latches win_0
